# Q3 · T4 · 2,000 → 6,000번 추가 학습

이전 모델의 **2,000번 체크포인트에서 추가 4,000번** 학습합니다.
같은 학습 128개·검증 32개 영상, 같은 모델·손실·학습률을 사용합니다.
모델·optimizer·AMP scaler·난수·step을 모두 이어받습니다.
32프레임 연속 예측을 계속 학습하며, 500번마다 같은 검증 영상에서 비교합니다.
이전 실험은 보존하고 새 폴더에 결과를 저장합니다. 실제 행동 라벨은 사용하지 않습니다.


## 1. 연속 학습 설정
`MAX_STEPS=6000`은 이번 단계의 누적 종료 지점입니다.
첫 실행은 이전 latest.pt에서 2,000 → 6,000번까지 학습합니다.
같은 새 실행 폴더의 latest.pt가 있으면 그 시점부터 남은 업데이트를 재개합니다.


In [ ]:
REPO_URL = "https://github.com/seungjoolee24/krafton-q3-video-worldmodel.git"
GIT_REF = "main"
PROFILE = "pilot"
RUN_NAME = "video-only-t4-6000-v1"
DRIVE_ROOT = "/content/drive/MyDrive/krafton-q3-video-worldmodel"
SOURCE_RUN_NAME = "video-only-t4-expanded-v1"
DATA_ZIP = DRIVE_ROOT + "/track3-kit.zip"
DATA_SHA256 = "dbd9da58b386782533913da2c44b088a6c25c6119ba6efbe680dae56bf4422ee"
MAX_STEPS = 6000
BATCH_SIZE = 4
print({"source_step": 2000, "stop_step": MAX_STEPS, "additional_updates": MAX_STEPS - 2000,
       "train_videos": 128, "dev_videos": 32, "training_horizon": 32, "actions_used": False})


## 2. 코드와 GPU 환경
현재 Git 커밋을 출력합니다. 재현할 때 `GIT_REF`에 이 SHA를 사용하세요.
Colab에 설치된 CUDA Torch를 유지하고 나머지 의존성만 설치합니다.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
version = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
assert version >= (2, 3), f"Torch >= 2.3 required, found {torch.__version__}"
assert PROFILE in {"pilot", "full"} and BATCH_SIZE >= 1 and MAX_STEPS >= 0
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME, "RUN_NAME은 폴더 이름 하나로 지정하세요."
CODE = Path("/content/q3-video-code")
if not (CODE / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(CODE), "remote", "get-url", "origin"], text=True).strip()
    assert remote == REPO_URL, "Existing checkout has a different repository."
    dirty = subprocess.check_output(["git", "-C", str(CODE), "status", "--porcelain"], text=True)
    assert not dirty.strip(), "Source changes exist; preserve them before changing revision."
subprocess.run(["git", "-C", str(CODE), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(CODE), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CODE), "--no-deps"], check=True)
# Editable-install .pth files are read by new processes, not the running kernel.
SOURCE_PATH = str(CODE / "src")
if SOURCE_PATH not in sys.path:
    sys.path.insert(0, SOURCE_PATH)
import importlib
importlib.invalidate_caches()
import video_wam
assert Path(video_wam.__file__).resolve().is_relative_to(CODE.resolve())
os.chdir(CODE)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
print({"commit": COMMIT, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)})
subprocess.run([sys.executable, "scripts/check_project.py"], check=True)


## 3. Drive 연결과 원본 데이터
Google Drive 접근은 본인 계정으로 승인합니다.
데이터 ZIP을 런타임 로컬 디스크로 복사한 뒤 압축을 풉니다.
학습 중 Drive에서 수천 개의 파일을 반복해서 읽지 않습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", timeout_ms=60000)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
source_zip = Path(DATA_ZIP)
LOCAL_DATA = Path("/content/q3-data")
LOCAL_DATA.mkdir(exist_ok=True)
local_zip = LOCAL_DATA / "track3-kit.zip"
from importlib.util import spec_from_file_location, module_from_spec
specification = spec_from_file_location("unpack_kit", CODE / "scripts/unpack_kit.py")
unpack_module = module_from_spec(specification)
specification.loader.exec_module(unpack_module)
unpack_module.stage_archive(source_zip, local_zip, DATA_SHA256)
KIT = unpack_module.unpack(local_zip, LOCAL_DATA / "unpacked")
CACHE = Path(f"/content/q3-cache-{PROFILE}")
LOCAL_RUN = Path(f"/content/q3-runs/{RUN_NAME}")
DRIVE_RUN = DRIVE / "runs" / RUN_NAME
config = json.loads((CODE / "configs" / f"{PROFILE}.json").read_text())
config["batch_size"] = BATCH_SIZE
config["eval_every"] = 500
config["checkpoint_every"] = 250
config["log_every"] = 50
config["evaluate_at_start"] = True
SOURCE_RUN = DRIVE / "runs" / SOURCE_RUN_NAME
if MAX_STEPS:
    config["total_steps"] = MAX_STEPS
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOCAL_RUN / "config.json"
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print({"kit": str(KIT), "cache": str(CACHE), "drive_results": str(DRIVE_RUN), "config": config})


## 4. GPU 동작 확인
기존 학습 코드와 데이터 구조를 그대로 사용합니다.
짧은 합성 GPU 동작 검사를 통과한 뒤 실제 체크포인트로 재개합니다.


In [ ]:
subprocess.run([sys.executable, "-m", "video_wam.cli", "smoke",
                "--kit", str(KIT), "--device", "cuda"], check=True)


## 5. 영상 특징 캐시
행동 sidecar는 읽지 않습니다. 원본 RGB를 제공된 고정 인코더로 변환합니다.
Pilot는 약 3GB, 전체는 약 35GB의 캐시 공간이 필요합니다.
중단된 같은 런타임에서 다시 실행하면 검증된 에피소드 캐시를 재사용합니다.
런타임이 삭제되면 원본 ZIP에서 다시 생성합니다.

In [ ]:
subprocess.run([sys.executable, "-m", "video_wam.cli", "prepare",
                "--kit", str(KIT), "--cache", str(CACHE), "--config", str(CONFIG_PATH),
                "--encode-batch", "64", "--device", "cuda"], check=True)
index = json.loads((CACHE / "index.json").read_text())
print({"train_episodes": len(index["train_ids"]), "dev_episodes": len(index["dev_ids"]),
       "actions_used": index["actions_used"], "cache_fingerprint": index["fingerprint"]})


## 6. 2,000 → 6,000번 학습
이전 실험의 최저 오차 모델과 2,000번 비교 결과를 새 폴더에도 보존합니다.
이번 추가 학습은 시작부터 끝까지 32프레임 예측을 사용합니다.
로그를 화면에 표시하고, 250번마다 저장하며 500번마다 32개 영상을 검증합니다.
중단 버튼을 누르면 마지막 완료된 업데이트의 latest.pt를 저장합니다.


In [ ]:
import signal
from video_wam.utils import persist_file, atomic_json
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
resume = DRIVE_RUN / "latest.pt"
if not resume.is_file():
    resume = SOURCE_RUN / "latest.pt"
    assert resume.is_file(), f"Missing source checkpoint: {resume}"
    source = torch.load(resume, map_location="cpu", weights_only=True)
    assert source["step"] == 2000, "Expected the completed 2,000-step pilot."
    assert source["cache_fingerprint"] == index["fingerprint"], "Selected data must match the original run."
    assert source["model_config"] == config["model"]
    source_best = torch.load(SOURCE_RUN / "best.pt", map_location="cpu", weights_only=True)
    assert source_best["cache_fingerprint"] == index["fingerprint"]
    if not (DRIVE_RUN / "best.pt").exists():
        persist_file(SOURCE_RUN / "best.pt", DRIVE_RUN)
    for step in sorted({source["step"], source_best["step"]}):
        source_report = SOURCE_RUN / "validation" / f"step_{step:06d}"
        assert (source_report / "metrics.json").is_file()
        destination = DRIVE_RUN / "validation" / source_report.name
        for artifact in source_report.iterdir():
            if artifact.is_file() and not (destination / artifact.name).exists():
                persist_file(artifact, destination)
    lineage = {"source_run": str(SOURCE_RUN), "source_step": source["step"],
               "source_git_revision": source["git_revision"], "target_step": MAX_STEPS,
               "optimizer_restored": True, "rng_restored": True,
               "cache_fingerprint": index["fingerprint"], "actions_used": False}
    atomic_json(LOCAL_RUN / "continuation.json", lineage)
    persist_file(LOCAL_RUN / "continuation.json", DRIVE_RUN)
    del source, source_best
command = [sys.executable, "-u", "-m", "video_wam.cli", "train",
           "--kit", str(KIT), "--cache", str(CACHE), "--config", str(CONFIG_PATH),
           "--out", str(LOCAL_RUN), "--persist", str(DRIVE_RUN),
           "--resume", str(resume), "--max-steps", str(MAX_STEPS), "--device", "cuda"]
print("Resume full training state:", resume, flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    process.wait(timeout=30)
    raise
if return_code:
    raise subprocess.CalledProcessError(return_code, command)


## 7. 추가 학습의 효과
같은 검증 영상 32개에서 추가 학습 전(2,000번)과 이후를 비교합니다.
새 실험에서 개선되지 않으면 이전의 best.pt가 그대로 유지됩니다.
물체 영역의 오차는 색 기반 보조 지표입니다. 비교 영상의 위치·각도·선명도도 함께 확인하세요.
런타임 초기화 후에는 이 결과 셀만 실행해 Drive의 완료 결과를 읽을 수 있습니다.


In [ ]:
import json
import torch
from pathlib import Path
from google.colab import drive
from IPython.display import display, HTML, Video
drive.mount("/content/drive", timeout_ms=60000)
if "DRIVE_RUN" not in globals():
    DRIVE_RUN = Path("/content/drive/MyDrive/krafton-q3-video-worldmodel/runs/video-only-t4-6000-v1")
    LOCAL_RUN = Path("/content/q3-runs/video-only-t4-6000-v1")
completion_path = LOCAL_RUN / "completion.json"
if not completion_path.exists():
    completion_path = DRIVE_RUN / "completion.json"
completion = json.loads(completion_path.read_text())

def report_for(step):
    relative = Path("validation") / f"step_{step:06d}"
    folder = LOCAL_RUN / relative
    if not (folder / "metrics.json").exists():
        folder = DRIVE_RUN / relative
    return folder, json.loads((folder / "metrics.json").read_text())

_, initial = report_for(2000)
_, final = report_for(completion["step"])
best_checkpoint = torch.load(DRIVE_RUN / "best.pt", map_location="cpu", weights_only=True)
best_folder, best = report_for(best_checkpoint["step"])
rows = [
    ("추가 학습 전 (2,000번)", initial["mse"]["mean"], initial["foreground_mse_heuristic"]["mean"]),
    (f"최종 모델 ({completion['step']:,}번)", final["mse"]["mean"], final["foreground_mse_heuristic"]["mean"]),
    (f"보존된 최소 오차 모델 ({best_checkpoint['step']:,}번)", best["mse"]["mean"], best["foreground_mse_heuristic"]["mean"]),
    ("copy-last 기준선", final["copy_last_mse"]["mean"], final["copy_last_foreground_mse_heuristic"]["mean"]),
]
table_rows = "".join(f"<tr><td>{name}</td><td>{rgb:.8f}</td><td>{fg:.6f}</td></tr>" for name, rgb, fg in rows)
display(HTML("<table><thead><tr><th>같은 검증 영상 32개 · 미래 32프레임</th>"
             "<th>RGB MSE ↓</th><th>물체 영역 MSE ↓</th></tr></thead><tbody>"
             + table_rows + "</tbody></table>"))
print(f"추가 학습 전 대비 최종 RGB 오차 감소: {(1 - final['mse']['mean'] / initial['mse']['mean']) * 100:.2f}%")
print(f"copy-last 대비 최종 RGB 오차 감소: {(1 - final['mse']['mean'] / final['copy_last_mse']['mean']) * 100:.2f}%")
print(f"이번 추가 업데이트: {completion['step'] - 2000:,}; 누적 업데이트: {completion['step']:,} / {completion['requested_steps']:,}")
print(f"학습·검증 시간: {completion['wall_seconds'] / 60:.2f}분; 종료: {completion['reason']}")
print(f"학습 GPU 최대 할당 메모리: {completion['gpu_peak_memory_gb']:.3f} GB; 실제 행동 라벨 사용: False")
print("시점별 최종 RGB MSE:", {key: final["mse"][key] for key in ["h1", "h8", "h16", "h32"]})
print("저장 위치:", DRIVE_RUN)
for video in sorted(best_folder.glob("*.mp4")):
    display(Video(str(video), embed=True, width=680))
